# 16. Economic subindex integration

## Purpose

Integrate three distinct economic dimensions into a signed country-year subindex:

1. merchandise trade orientation;
2. FDI structural orientation; and
3. economic institutional orientation.

All component scores use the same direction: positive values indicate greater closeness to the United States and negative values indicate greater closeness to China.

This notebook is a development and validation record. It does not write production outputs.

## FDI reporting architecture

Each country is assigned one fixed official FDI reporting regime. Countries with sufficiently sustained host-reported official coverage use that perspective throughout 2009–2024. All other countries use anchor-reported official positions during 2018–2024. No country switches reporting perspective over time.

The preferred host assignment requires signed host-official positions in at least six of nine years during 2009–2017 and at least four of seven years during 2018–2024. Moderate and strict availability thresholds remain sensitivities.

## Period architecture

- **2004–2008:** pre-FDI source period. Trade and institutional ties may contribute.
- **2009–2017:** fixed-regime historical extension. Host-regime countries may have FDI; anchor-regime countries do not.
- **2018–2024:** fixed-regime common period. Both reporting regimes may produce FDI scores without within-country perspective switching.
- **2025:** provisional latest-year extension. FDI is unavailable; trade coverage is incomplete; institutional states were analyst-verified as unchanged from 2024.

At least two economic components are required to calculate an economic score. When two or three components are available, equal conceptual weights are renormalised across the available components. Zero filling is retained only as a sensitivity. Composition and period flags prevent observations with different evidence bases from being presented as identical.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
TRADE_PATH = ROOT / "data/processed/trade_orientation_country_year.csv"
FDI_PATH = ROOT / "data/processed/fdi_structural_ties_country_anchor_year.csv"
INSTITUTIONAL_PATH = ROOT / "data/processed/economic_institutional_ties_country_anchor_year.csv"
FDI_REGIME_SENSITIVITY_PATH = ROOT / "outputs/tables/fdi/fdi_regime_assignment_sensitivity_by_year.csv"

trade_raw = pd.read_csv(TRADE_PATH, low_memory=False)
fdi_raw = pd.read_csv(FDI_PATH, low_memory=False)
institutional_raw = pd.read_csv(INSTITUTIONAL_PATH, low_memory=False)
fdi_regime_sensitivity = pd.read_csv(FDI_REGIME_SENSITIVITY_PATH)

pd.Series({
    "trade_rows": len(trade_raw),
    "fdi_rows": len(fdi_raw),
    "institutional_rows": len(institutional_raw),
    "fdi_regime_sensitivity_rows": len(fdi_regime_sensitivity),
}, name="value")

trade_rows                     4205
fdi_rows                       8410
institutional_rows             8410
fdi_regime_sensitivity_rows      66
Name: value, dtype: int64

## 1. Validate and reshape the three production components

In [2]:
if trade_raw.duplicated(["country_iso3", "year"]).any():
    raise ValueError("Duplicate trade country-year keys detected.")
if fdi_raw.duplicated(["country_iso3", "anchor", "year"]).any():
    raise ValueError("Duplicate FDI country-anchor-year keys detected.")
if institutional_raw.duplicated(["country_iso3", "anchor", "year"]).any():
    raise ValueError("Duplicate institutional country-anchor-year keys detected.")

if not trade_raw["trade_orientation_score"].dropna().between(-1, 1).all():
    raise ValueError("Trade scores fall outside [-1, 1].")
if not fdi_raw["fdi_structural_score_raw"].dropna().between(0, 1).all():
    raise ValueError("FDI anchor scores fall outside [0, 1].")
if not institutional_raw["economic_institutional_score_raw"].between(0, 1).all():
    raise ValueError("Institutional anchor scores fall outside [0, 1].")

fdi = fdi_raw.pivot(
    index=["country_iso3", "preferred_country_name", "year"],
    columns="anchor",
    values="fdi_structural_score_raw",
).reset_index()
fdi["fdi_orientation_score"] = fdi["United States"] - fdi["China"]

regime_columns = [
    "country_iso3", "fdi_reporting_regime", "regime_assignment_rule",
    "host_signed_years_2009_2017", "host_signed_years_2018_2024",
]
regime_audit = fdi_raw.drop_duplicates("country_iso3")[regime_columns]
fdi = fdi.merge(regime_audit, on="country_iso3", how="left", validate="many_to_one")

institutional = institutional_raw.pivot(
    index=["country_iso3", "preferred_country_name", "year"],
    columns="anchor",
    values="economic_institutional_score_raw",
).reset_index()
institutional["economic_institutional_orientation_score"] = (
    institutional["United States"] - institutional["China"]
)

if not fdi["fdi_orientation_score"].dropna().between(-1, 1).all():
    raise ValueError("FDI orientation falls outside [-1, 1].")
if not institutional["economic_institutional_orientation_score"].dropna().between(-1, 1).all():
    raise ValueError("Institutional orientation falls outside [-1, 1].")

regime_counts = regime_audit.loc[
    regime_audit["country_iso3"].ne("SCG"), "fdi_reporting_regime"
].value_counts()
if regime_counts.to_dict() != {
    "anchor_official_fixed_regime": 112,
    "host_official_fixed_regime": 79,
}:
    raise ValueError(f"Unexpected FDI regime counts: {regime_counts.to_dict()}")

pd.DataFrame({
    "component": ["trade", "fdi", "institutional"],
    "rows": [len(trade_raw), len(fdi), len(institutional)],
    "available_scores": [
        trade_raw["trade_orientation_score"].notna().sum(),
        fdi["fdi_orientation_score"].notna().sum(),
        institutional["economic_institutional_orientation_score"].notna().sum(),
    ],
})

,component,rows,available_scores
0,trade,4205,3465
1,fdi,4205,1834
2,institutional,4205,4205


## 2. Join the components on the canonical country-year universe

All three production panels are expected to use the same 4,205-row canonical country-year grid.

In [3]:
trade = trade_raw[[
    "country_iso3", "preferred_country_name", "year",
    "trade_orientation_score", "trade_evidence_status",
    "trade_provenance_status", "trade_comparability_status",
]].copy()

integrated = trade.merge(
    fdi[[
        "country_iso3", "year", "fdi_orientation_score",
        "fdi_reporting_regime", "regime_assignment_rule",
        "host_signed_years_2009_2017", "host_signed_years_2018_2024",
    ]],
    on=["country_iso3", "year"], how="outer", validate="one_to_one",
    indicator="trade_fdi_join",
).merge(
    institutional[[
        "country_iso3", "year", "economic_institutional_orientation_score"
    ]],
    on=["country_iso3", "year"], how="outer", validate="one_to_one",
    indicator="institutional_join",
)

if not integrated["trade_fdi_join"].eq("both").all():
    raise ValueError(integrated["trade_fdi_join"].value_counts().to_dict())
if not integrated["institutional_join"].eq("both").all():
    raise ValueError(integrated["institutional_join"].value_counts().to_dict())

integrated = integrated.drop(columns=["trade_fdi_join", "institutional_join"])
component_columns = [
    "trade_orientation_score",
    "fdi_orientation_score",
    "economic_institutional_orientation_score",
]
integrated["available_component_count"] = integrated[component_columns].notna().sum(axis=1)

pd.Series({
    "rows": len(integrated),
    "entities": integrated["country_iso3"].nunique(),
    "first_year": integrated["year"].min(),
    "last_year": integrated["year"].max(),
    "duplicate_country_years": integrated.duplicated(["country_iso3", "year"]).sum(),
}, name="value")

rows                       4205
entities                    192
first_year                 2004
last_year                  2025
duplicate_country_years       0
Name: value, dtype: int64

## 3. Construct the minimum-two-component equal-weight baseline

When all three components are available, each receives one-third weight. When exactly two components are available, the remaining conceptual weights are renormalised equally. An institutional-only observation is not sufficient to represent the broader economic relationship and remains score-unavailable.

In [4]:
MINIMUM_ECONOMIC_COMPONENTS = 2

trade_available = integrated["trade_orientation_score"].notna()
fdi_available = integrated["fdi_orientation_score"].notna()
institutional_available = integrated["economic_institutional_orientation_score"].notna()
all_three_available = trade_available & fdi_available & institutional_available
two_components_available = integrated["available_component_count"].eq(2)

integrated["economic_score_baseline"] = integrated[component_columns].mean(
    axis=1, skipna=True
).where(integrated["available_component_count"].ge(MINIMUM_ECONOMIC_COMPONENTS))

integrated["economic_component_status"] = np.select(
    [
        all_three_available,
        trade_available & ~fdi_available & institutional_available,
        ~trade_available & fdi_available & institutional_available,
        ~trade_available & ~fdi_available & institutional_available,
    ],
    [
        "trade_fdi_institutional_available",
        "trade_and_institutional_available_fdi_unavailable",
        "fdi_and_institutional_available_trade_unavailable",
        "institutional_only_score_unavailable",
    ],
    default="unexpected_component_pattern",
)

integrated["economic_comparability_status"] = np.select(
    [
        integrated["year"].between(2004, 2008) & two_components_available,
        integrated["year"].between(2004, 2008) & integrated["available_component_count"].eq(1),
        integrated["year"].between(2009, 2017) & all_three_available,
        integrated["year"].between(2009, 2017) & two_components_available,
        integrated["year"].between(2009, 2017) & integrated["available_component_count"].eq(1),
        integrated["year"].between(2018, 2024) & all_three_available,
        integrated["year"].between(2018, 2024) & two_components_available,
        integrated["year"].between(2018, 2024) & integrated["available_component_count"].eq(1),
        integrated["year"].eq(2025) & two_components_available,
        integrated["year"].eq(2025) & integrated["available_component_count"].eq(1),
    ],
    [
        "pre_fdi_source_period_two_component_extension",
        "pre_fdi_source_period_score_unavailable",
        "fixed_regime_historical_extension_three_components",
        "fixed_regime_historical_extension_two_components",
        "fixed_regime_historical_extension_score_unavailable",
        "fixed_regime_common_period_three_components",
        "fixed_regime_common_period_two_components",
        "fixed_regime_common_period_score_unavailable",
        "provisional_latest_year_two_component_extension",
        "provisional_latest_year_score_unavailable",
    ],
    default="invalid",
)

if not integrated["economic_score_baseline"].dropna().between(-1, 1).all():
    raise ValueError("Economic baseline scores fall outside [-1, 1].")

status_summary = integrated.groupby(
    ["economic_comparability_status", "economic_component_status"], dropna=False
).agg(
    country_years=("country_iso3", "size"),
    countries=("country_iso3", "nunique"),
    mean_trade=("trade_orientation_score", "mean"),
    mean_fdi=("fdi_orientation_score", "mean"),
    mean_institutional=("economic_institutional_orientation_score", "mean"),
    mean_economic=("economic_score_baseline", "mean"),
).reset_index()
status_summary["share"] = status_summary["country_years"] / len(integrated)
status_summary

,economic_comparability_status,economic_component_status,country_years,countries,mean_trade,mean_fdi,mean_institutional,mean_economic,share
0,fixed_regime_common_period_score_unavailable,institutional_only_score_unavailable,41,17,NaN,NaN,-0.094817,NaN,0.009750
1,fixed_regime_common_period_three_components,trade_fdi_institutional_available,971,166,-0.253299,-0.067920,-0.112397,-0.144539,0.230916
2,fixed_regime_common_period_two_components,fdi_and_institutional_available_trade_unavailable,200,50,NaN,-0.181878,-0.076187,-0.129033,0.047562
3,fixed_regime_common_period_two_components,trade_and_institutional_available_fdi_unavailable,125,43,-0.254991,NaN,-0.092100,-0.173545,0.029727
4,fixed_regime_historical_extension_score_unavai...,institutional_only_score_unavailable,248,43,NaN,NaN,-0.075101,NaN,0.058977
5,fixed_regime_historical_extension_three_compon...,trade_fdi_institutional_available,658,79,-0.195021,0.097434,-0.145080,-0.080889,0.156480
6,fixed_regime_historical_extension_two_components,fdi_and_institutional_available_trade_unavailable,5,2,NaN,-0.239515,-0.052500,-0.146008,0.001189
7,fixed_regime_historical_extension_two_components,trade_and_institutional_available_fdi_unavailable,808,124,-0.041123,NaN,-0.055198,-0.048160,0.192152
8,pre_fdi_source_period_score_unavailable,institutional_only_score_unavailable,177,51,NaN,NaN,-0.061370,NaN,0.042093
9,pre_fdi_source_period_two_component_extension,trade_and_institutional_available_fdi_unavailable,781,169,0.146322,NaN,-0.070391,0.037966,0.185731


## 4. Annual component and FDI-regime coverage

In [5]:
coverage_by_year = integrated.groupby("year").agg(
    eligible_country_years=("country_iso3", "size"),
    trade_scores=("trade_orientation_score", "count"),
    fdi_scores=("fdi_orientation_score", "count"),
    institutional_scores=("economic_institutional_orientation_score", "count"),
    all_three_scores=("available_component_count", lambda values: int(values.eq(3).sum())),
    two_component_scores=("available_component_count", lambda values: int(values.eq(2).sum())),
    one_component_unavailable=("available_component_count", lambda values: int(values.eq(1).sum())),
    economic_scores=("economic_score_baseline", "count"),
).reset_index()
for column in ["trade_scores", "fdi_scores", "institutional_scores", "all_three_scores", "economic_scores"]:
    coverage_by_year[f"{column}_rate"] = coverage_by_year[column] / coverage_by_year["eligible_country_years"]

display(coverage_by_year)

fdi_coverage_by_regime = (
    integrated.loc[integrated["fdi_orientation_score"].notna()]
    .groupby(["year", "fdi_reporting_regime"])
    .size()
    .rename("signed_fdi_scores")
    .reset_index()
)
display(fdi_coverage_by_regime)

fdi_regime_sensitivity

,year,eligible_country_years,trade_scores,fdi_scores,institutional_scores,all_three_scores,two_component_scores,one_component_unavailable,economic_scores,trade_scores_rate,fdi_scores_rate,institutional_scores_rate,all_three_scores_rate,economic_scores_rate
0,2004,192,154,0,192,0,154,38,154,0.802083,0.000000,1.0,0.000000,0.802083
1,2005,192,154,0,192,0,154,38,154,0.802083,0.000000,1.0,0.000000,0.802083
2,2006,192,157,0,192,0,157,35,157,0.817708,0.000000,1.0,0.000000,0.817708
3,2007,191,160,0,191,0,160,31,160,0.837696,0.000000,1.0,0.000000,0.837696
4,2008,191,156,0,191,0,156,35,156,0.816754,0.000000,1.0,0.000000,0.816754
5,2009,191,160,63,191,62,99,30,161,0.837696,0.329843,1.0,0.324607,0.842932
6,2010,191,165,67,191,66,100,25,166,0.863874,0.350785,1.0,0.345550,0.869110
7,2011,191,162,72,191,71,92,28,163,0.848168,0.376963,1.0,0.371728,0.853403
8,2012,191,160,76,191,75,86,30,161,0.837696,0.397906,1.0,0.392670,0.842932
9,2013,191,162,78,191,78,84,29,162,0.848168,0.408377,1.0,0.408377,0.848168


,year,fdi_reporting_regime,signed_fdi_scores
0,2009,host_official_fixed_regime,63
1,2010,host_official_fixed_regime,67
2,2011,host_official_fixed_regime,72
3,2012,host_official_fixed_regime,76
4,2013,host_official_fixed_regime,78
5,2014,host_official_fixed_regime,79
6,2015,host_official_fixed_regime,76
7,2016,host_official_fixed_regime,74
8,2017,host_official_fixed_regime,78
9,2018,anchor_official_fixed_regime,92


,assignment_rule,pre_threshold,post_threshold,year,host_regime_countries,anchor_regime_countries,signed_fdi_scores
0,moderate,5,4,2004,84,107,0
1,moderate,5,4,2005,84,107,0
2,moderate,5,4,2006,84,107,0
3,moderate,5,4,2007,84,107,0
4,moderate,5,4,2008,84,107,0
...,...,...,...,...,...,...,...
61,strict,7,5,2021,70,121,169
62,strict,7,5,2022,70,121,167
63,strict,7,5,2023,70,121,163
64,strict,7,5,2024,70,121,161


## 5. Missingness sensitivity: renormalisation versus zero filling

In [6]:
integrated["economic_score_zero_fill"] = integrated[component_columns].fillna(0).mean(axis=1)
comparable_for_missingness = integrated["economic_score_baseline"].notna()
missingness_change = (
    integrated.loc[comparable_for_missingness, "economic_score_baseline"]
    - integrated.loc[comparable_for_missingness, "economic_score_zero_fill"]
).abs()
affected = missingness_change.gt(1e-12)

pd.Series({
    "scored_country_years": int(comparable_for_missingness.sum()),
    "affected_country_years": int(affected.sum()),
    "mean_absolute_change_among_affected": missingness_change[affected].mean(),
    "maximum_absolute_change": missingness_change[affected].max(),
    "sign_changes": int((
        np.sign(integrated.loc[missingness_change.index[affected], "economic_score_baseline"])
        != np.sign(integrated.loc[missingness_change.index[affected], "economic_score_zero_fill"])
    ).sum()),
}, name="value")

scored_country_years                   3670.000000
affected_country_years                 2017.000000
mean_absolute_change_among_affected       0.085333
maximum_absolute_change                   0.247134
sign_changes                              0.000000
Name: value, dtype: float64

## 6. Component-weight sensitivities

Compare equal weights with two plausible alternatives on common three-component support:

- **real-economy emphasis:** 40% trade, 40% FDI, 20% institutional ties;
- **institutional emphasis:** 25% trade, 25% FDI, 50% institutional ties.

Outside common support, each specification renormalises its stated weights across the available components.

In [7]:
def weighted_available_score(frame, weights):
    values = frame[list(weights)].copy()
    availability = values.notna()
    weighted = values.mul(pd.Series(weights))
    denominator = availability.mul(pd.Series(weights)).sum(axis=1)
    return weighted.sum(axis=1, skipna=True).div(denominator).where(denominator.gt(0))

integrated["economic_score_real_economy_emphasis"] = weighted_available_score(
    integrated,
    {
        "trade_orientation_score": 0.40,
        "fdi_orientation_score": 0.40,
        "economic_institutional_orientation_score": 0.20,
    },
)
integrated["economic_score_institutional_emphasis"] = weighted_available_score(
    integrated,
    {
        "trade_orientation_score": 0.25,
        "fdi_orientation_score": 0.25,
        "economic_institutional_orientation_score": 0.50,
    },
)

rows = []
for label, column in {
    "real_economy_emphasis": "economic_score_real_economy_emphasis",
    "institutional_emphasis": "economic_score_institutional_emphasis",
}.items():
    valid = all_three_available
    change = (integrated.loc[valid, column] - integrated.loc[valid, "economic_score_baseline"]).abs()
    rows.append({
        "specification": label,
        "common_support_country_years": int(valid.sum()),
        "mean_absolute_change": change.mean(),
        "maximum_absolute_change": change.max(),
        "sign_changes": int((
            np.sign(integrated.loc[valid, column])
            != np.sign(integrated.loc[valid, "economic_score_baseline"])
        ).sum()),
    })

pd.DataFrame(rows)

,specification,common_support_country_years,mean_absolute_change,maximum_absolute_change,sign_changes
0,real_economy_emphasis,1629,0.038870,0.142745,60
1,institutional_emphasis,1629,0.048587,0.178432,54


## 7. Component relationships and disagreement diagnostics

In [8]:
display(integrated[component_columns].corr())

common = integrated.loc[all_three_available].copy()
for component in component_columns:
    common[f"gap_{component}"] = (
        common[component] - common["economic_score_baseline"]
    ).abs()
common["maximum_component_gap"] = common[[
    f"gap_{component}" for component in component_columns
]].max(axis=1)
common["component_sign_pattern"] = (
    np.sign(common["trade_orientation_score"]).astype(int).astype(str)
    + "|" + np.sign(common["fdi_orientation_score"]).astype(int).astype(str)
    + "|" + np.sign(common["economic_institutional_orientation_score"]).astype(int).astype(str)
)

display(common["component_sign_pattern"].value_counts().rename_axis(
    "trade|fdi|institutional"
).reset_index(name="country_years"))

common.sort_values("maximum_component_gap", ascending=False)[[
    "country_iso3", "preferred_country_name", "year",
    *component_columns, "economic_score_baseline",
    "component_sign_pattern", "maximum_component_gap",
]].head(25)

,trade_orientation_score,fdi_orientation_score,economic_institutional_orientation_score
trade_orientation_score,1.000000,0.416250,0.172782
fdi_orientation_score,0.416250,1.000000,0.168362
economic_institutional_orientation_score,0.172782,0.168362,1.000000


,trade|fdi|institutional,country_years
0,-1|-1|-1,426
1,-1|1|0,271
2,-1|-1|0,233
3,-1|1|-1,221
4,1|1|-1,148
5,1|1|1,85
6,1|1|0,46
7,1|-1|0,44
8,1|-1|-1,40
9,-1|-1|1,27


,country_iso3,preferred_country_name,year,trade_orientation_score,fdi_orientation_score,economic_institutional_orientation_score,economic_score_baseline,component_sign_pattern,maximum_component_gap
526,BRA,Brazil,2024,-0.321029,0.971538,0.0000,0.216836,-1|1|0,0.754701
1635,HUN,Hungary,2011,-0.296174,0.823926,-0.2500,0.092584,-1|1|-1,0.731342
84,AND,Andorra,2022,-0.999961,0.161205,0.0000,-0.279585,-1|1|0,0.720376
81,AND,Andorra,2019,-0.999917,0.159701,0.0000,-0.280072,-1|1|0,0.719845
85,AND,Andorra,2023,-0.999255,0.158169,0.0000,-0.280362,-1|1|0,0.718893
86,AND,Andorra,2024,-0.999014,0.157993,0.0000,-0.280341,-1|1|0,0.718674
83,AND,Andorra,2021,-0.996790,0.158928,0.0000,-0.279287,-1|1|0,0.717503
3689,TGO,Togo,2016,-0.848094,0.449128,0.0000,-0.132989,-1|1|0,0.715105
1779,ISL,Iceland,2023,0.170868,0.570311,-0.7000,0.013726,1|1|-1,0.713726
1643,HUN,Hungary,2019,-0.202914,0.813398,-0.2500,0.120161,-1|1|-1,0.693237


## 8. Rank sensitivity on common three-component support

In [9]:
rank_rows = []
for year, group in integrated.loc[all_three_available].groupby("year"):
    group = group.copy()
    group["baseline_rank"] = group["economic_score_baseline"].rank(
        ascending=False, method="average"
    )
    for label, column in {
        "real_economy_emphasis": "economic_score_real_economy_emphasis",
        "institutional_emphasis": "economic_score_institutional_emphasis",
    }.items():
        alternative_rank = group[column].rank(ascending=False, method="average")
        displacement = (alternative_rank - group["baseline_rank"]).abs()
        rank_rows.append({
            "year": int(year),
            "specification": label,
            "countries": int(len(group)),
            "mean_absolute_rank_change": displacement.mean(),
            "maximum_absolute_rank_change": displacement.max(),
            "top_10_membership_changes": int(len(
                set(group.loc[group["baseline_rank"].le(10), "country_iso3"])
                ^ set(group.loc[alternative_rank.le(10), "country_iso3"])
            )),
        })

pd.DataFrame(rank_rows)

,year,specification,countries,mean_absolute_rank_change,maximum_absolute_rank_change,top_10_membership_changes
0,2009,real_economy_emphasis,62,2.161290,10.0,0
1,2009,institutional_emphasis,62,4.032258,16.0,2
2,2010,real_economy_emphasis,66,3.242424,10.0,0
3,2010,institutional_emphasis,66,5.242424,21.0,2
4,2011,real_economy_emphasis,71,2.647887,9.0,0
5,2011,institutional_emphasis,71,4.957746,19.0,0
6,2012,real_economy_emphasis,75,3.253333,9.0,0
7,2012,institutional_emphasis,75,5.466667,22.0,4
8,2013,real_economy_emphasis,78,3.256410,14.0,2
9,2013,institutional_emphasis,78,5.282051,18.0,2


## 9. Distribution and selected-country trajectories

Annual summaries retain period-composition labels. The 2004–2008, 2009–2017, 2018–2024, and 2025 segments must not be interpreted as compositionally identical.

In [10]:
display(integrated["economic_score_baseline"].describe())

annual_summary = integrated.groupby("year").agg(
    available_scores=("economic_score_baseline", "count"),
    mean_score=("economic_score_baseline", "mean"),
    median_score=("economic_score_baseline", "median"),
    min_score=("economic_score_baseline", "min"),
    max_score=("economic_score_baseline", "max"),
    mean_available_components=("available_component_count", "mean"),
).reset_index()
annual_summary["period_status"] = np.select(
    [
        annual_summary["year"].between(2004, 2008),
        annual_summary["year"].between(2009, 2017),
        annual_summary["year"].between(2018, 2024),
        annual_summary["year"].eq(2025),
    ],
    [
        "pre_fdi_source_period",
        "fixed_regime_historical_extension",
        "fixed_regime_common_period",
        "provisional_latest_year_extension",
    ],
    default="invalid",
)
annual_summary

count    3670.000000
mean       -0.075265
std         0.273850
min        -0.817394
25%        -0.271320
50%        -0.109773
75%         0.088863
max         0.712018
Name: economic_score_baseline, dtype: float64

,year,available_scores,mean_score,median_score,min_score,max_score,mean_available_components,period_status
0,2004,154,0.087404,0.091196,-0.612733,0.595733,1.802083,pre_fdi_source_period
1,2005,154,0.080714,0.048740,-0.599652,0.584818,1.802083,pre_fdi_source_period
2,2006,157,0.035998,0.003800,-0.610541,0.712018,1.817708,pre_fdi_source_period
3,2007,160,0.007726,-0.045025,-0.615946,0.686101,1.837696,pre_fdi_source_period
4,2008,156,-0.020042,-0.063008,-0.617721,0.550271,1.816754,pre_fdi_source_period
5,2009,161,-0.003523,-0.035780,-0.585832,0.662115,2.167539,fixed_regime_historical_extension
6,2010,166,-0.046227,-0.080619,-0.732746,0.615456,2.214660,fixed_regime_historical_extension
7,2011,163,-0.039059,-0.072456,-0.704495,0.622356,2.225131,fixed_regime_historical_extension
8,2012,161,-0.032576,-0.056097,-0.736568,0.620665,2.235602,fixed_regime_historical_extension
9,2013,162,-0.052351,-0.092655,-0.738312,0.593082,2.256545,fixed_regime_historical_extension


In [11]:
selected_years = [2004, 2008, 2009, 2017, 2018, 2020, 2024, 2025]
trajectory_iso3 = ["AUS", "BRA", "IND", "JPN", "PAK", "RUS", "VNM"]
integrated.loc[
    integrated["country_iso3"].isin(trajectory_iso3)
    & integrated["year"].isin(selected_years),
    [
        "country_iso3", "preferred_country_name", "year",
        "fdi_reporting_regime", *component_columns,
        "available_component_count", "economic_score_baseline",
        "economic_comparability_status",
    ],
].sort_values(["country_iso3", "year"])

,country_iso3,preferred_country_name,year,fdi_reporting_regime,trade_orientation_score,fdi_orientation_score,economic_institutional_orientation_score,available_component_count,economic_score_baseline,economic_comparability_status
176,AUS,Australia,2004,host_official_fixed_regime,0.021029,NaN,-0.2500,2,-0.114486,pre_fdi_source_period_two_component_extension
180,AUS,Australia,2008,host_official_fixed_regime,-0.261360,NaN,0.2000,2,-0.030680,pre_fdi_source_period_two_component_extension
181,AUS,Australia,2009,host_official_fixed_regime,-0.410913,-0.066543,0.2000,3,-0.092485,fixed_regime_historical_extension_three_compon...
189,AUS,Australia,2017,host_official_fixed_regime,-0.574106,-0.041889,-0.2500,3,-0.288665,fixed_regime_historical_extension_three_compon...
190,AUS,Australia,2018,host_official_fixed_regime,-0.594257,-0.011497,-0.2500,3,-0.285251,fixed_regime_common_period_three_components
192,AUS,Australia,2020,host_official_fixed_regime,-0.604668,-0.018431,-0.2500,3,-0.291033,fixed_regime_common_period_three_components
196,AUS,Australia,2024,host_official_fixed_regime,-0.565812,-0.034357,-0.5125,3,-0.370890,fixed_regime_common_period_three_components
197,AUS,Australia,2025,host_official_fixed_regime,-0.516438,NaN,-0.5125,2,-0.514469,provisional_latest_year_two_component_extension
506,BRA,Brazil,2004,host_official_fixed_regime,0.546877,NaN,0.0000,2,0.273439,pre_fdi_source_period_two_component_extension
510,BRA,Brazil,2008,host_official_fixed_regime,0.171493,NaN,0.0000,2,0.085747,pre_fdi_source_period_two_component_extension


## 10. Final validation and prospective production schema

The production candidate preserves each component and its composition flags. Sensitivity specifications and disagreement cases belong in audit outputs rather than the principal production panel.

In [12]:
production_columns = [
    "country_iso3", "preferred_country_name", "year",
    "fdi_reporting_regime", "regime_assignment_rule",
    *component_columns,
    "available_component_count", "economic_component_status",
    "economic_comparability_status", "economic_score_baseline",
]
production_candidate = integrated[production_columns].copy()

expected_component_counts = {1: 535, 2: 2041, 3: 1629}
actual_component_counts = production_candidate[
    "available_component_count"
].value_counts().sort_index().to_dict()
if actual_component_counts != expected_component_counts:
    raise ValueError(f"Unexpected component-count distribution: {actual_component_counts}")

validation = pd.Series({
    "rows": len(production_candidate),
    "entities": production_candidate["country_iso3"].nunique(),
    "first_year": production_candidate["year"].min(),
    "last_year": production_candidate["year"].max(),
    "duplicate_country_years": production_candidate.duplicated(["country_iso3", "year"]).sum(),
    "available_economic_scores": production_candidate["economic_score_baseline"].notna().sum(),
    "missing_economic_scores": production_candidate["economic_score_baseline"].isna().sum(),
    "score_min": production_candidate["economic_score_baseline"].min(),
    "score_max": production_candidate["economic_score_baseline"].max(),
    "three_component_scores": production_candidate["available_component_count"].eq(3).sum(),
    "two_component_scores": production_candidate["available_component_count"].eq(2).sum(),
    "one_component_unavailable_rows": production_candidate["available_component_count"].eq(1).sum(),
    "2025_fdi_scores_present": production_candidate.loc[
        production_candidate["year"].eq(2025), "fdi_orientation_score"
    ].notna().sum(),
})

if validation["rows"] != 4205 or validation["entities"] != 192:
    raise ValueError(f"Unexpected production dimensions: {validation.to_dict()}")
if validation["duplicate_country_years"] != 0:
    raise ValueError("Duplicate production country-year keys detected.")
if validation["available_economic_scores"] != 3670 or validation["missing_economic_scores"] != 535:
    raise ValueError("Unexpected economic-score availability.")
if validation["three_component_scores"] != 1629 or validation["two_component_scores"] != 2041:
    raise ValueError("Unexpected scored component composition.")
if validation["one_component_unavailable_rows"] != 535:
    raise ValueError("Unexpected one-component unavailable count.")
if not production_candidate.loc[
    production_candidate["available_component_count"].eq(1), "economic_score_baseline"
].isna().all():
    raise ValueError("One-component economic observations must remain unavailable.")
if not production_candidate.loc[
    production_candidate["available_component_count"].ge(2), "economic_score_baseline"
].notna().all():
    raise ValueError("Two- and three-component economic observations must be scored.")
if validation["2025_fdi_scores_present"] != 0:
    raise ValueError("FDI scores should remain unavailable in 2025.")
if not production_candidate["economic_score_baseline"].dropna().between(-1, 1).all():
    raise ValueError("Economic scores fall outside [-1, 1].")

validation

rows                              4205.000000
entities                           192.000000
first_year                        2004.000000
last_year                         2025.000000
duplicate_country_years              0.000000
available_economic_scores         3670.000000
missing_economic_scores            535.000000
score_min                           -0.817394
score_max                            0.712018
three_component_scores            1629.000000
two_component_scores              2041.000000
one_component_unavailable_rows     535.000000
2025_fdi_scores_present              0.000000
dtype: float64

## 11. Decision questions

Use the executed diagnostics to decide:

- whether equal weighting becomes the economic baseline;
- whether the minimum-two-component rule and available-component renormalisation become production rules;
- whether the preferred 6-of-9 and 4-of-7 fixed FDI regime assignment is retained;
- how prominently to publish the 2004–2008 pre-FDI segment and 2009–2017 fixed-regime historical extension;
- whether 2025 remains a provisional trade-plus-institutional extension;
- which component-weight and FDI-regime threshold alternatives remain formal sensitivities.

The prospective production output is `data/processed/economic_subindex_country_year.csv`. No production file is written by this notebook.